# Eksperimen MSML - Klasifikasi Gambar PlantVillage

Notebook ini disusun untuk memenuhi **Kriteria 1: Melakukan Eksperimen terhadap Dataset Pelatihan** pada submission MSML. Isi notebook sudah disesuaikan untuk kasus **klasifikasi gambar** dan dapat dijalankan di **VS Code + WSL + Conda** tanpa bergantung pada Google Colab.

Notebook ini mencakup data loading, EDA gambar, validasi gambar rusak, preprocessing manual, pembagian dataset `train`, `val`, `test`, serta penyimpanan `labels.txt` dan `dataset_metadata.json`.

> Jalankan notebook ini dari root repository `Eksperimen_SML_Nama-siswa` atau dari folder `preprocessing`. Kode dibuat fleksibel untuk dua posisi kerja tersebut.

# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **PlantVillage**, yaitu dataset citra daun tanaman yang memuat berbagai kelas penyakit tanaman dan kelas sehat. Dataset ini digunakan untuk membangun model klasifikasi gambar berbasis deep learning. Setiap folder kelas merepresentasikan label target, sedangkan setiap gambar di dalam folder tersebut menjadi sampel data.

Struktur dataset raw yang disarankan:

```text
Eksperimen_SML_Nama-siswa
├── namadataset_raw
│   └── PlantVillage.zip
└── preprocessing
    └── Eksperimen_Nama-siswa.ipynb
```

atau:

```text
Eksperimen_SML_Nama-siswa
├── namadataset_raw
│   └── PlantVillage
│       └── data
│           ├── Class_1
│           ├── Class_2
│           └── Class_n
└── preprocessing
    └── Eksperimen_Nama-siswa.ipynb
```

# **2. Import Library**

Library pada notebook ini digunakan untuk eksperimen dan preprocessing dataset gambar. TensorFlow hanya dipakai pada bagian akhir untuk memeriksa apakah dataset hasil preprocessing sudah dapat dibaca sebagai dataset training.

In [ ]:
import os
import json
import shutil
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False

try:
    import tensorflow as tf
    TENSORFLOW_AVAILABLE = True
except Exception as error:
    print("TensorFlow belum tersedia atau gagal di-import:", error)
    TENSORFLOW_AVAILABLE = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if TENSORFLOW_AVAILABLE:
    tf.random.set_seed(SEED)

print("Import library selesai.")

# **3. Konfigurasi Project**

Bagian ini mengatur lokasi dataset raw dan lokasi output preprocessing. Ganti `STUDENT_NAME` sesuai nama kamu sebelum notebook disimpan final.

In [ ]:
STUDENT_NAME = "Nama-siswa"          # Ganti dengan nama kamu
DATASET_NAME = "PlantVillage"        # Ganti jika nama dataset berbeda

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "preprocessing":
    PROJECT_ROOT = CURRENT_DIR.parent
    PREPROCESSING_DIR = CURRENT_DIR
else:
    PROJECT_ROOT = CURRENT_DIR
    PREPROCESSING_DIR = PROJECT_ROOT / "preprocessing"

RAW_DIR = PROJECT_ROOT / "namadataset_raw"
WORKING_DIR = PREPROCESSING_DIR / "_working_dataset"
OUTPUT_DIR = PREPROCESSING_DIR / "namadataset_preprocessing"

TRAIN_DIR = OUTPUT_DIR / "train"
VAL_DIR = OUTPUT_DIR / "val"
TEST_DIR = OUTPUT_DIR / "test"

SUPPORTED_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SUPPORTED_ARCHIVE_EXTENSIONS = {".zip"}

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

IMG_SIZE = 224
BATCH_SIZE = 16

for directory in [RAW_DIR, PREPROCESSING_DIR, WORKING_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root        :", PROJECT_ROOT)
print("Raw dataset folder  :", RAW_DIR)
print("Working folder      :", WORKING_DIR)
print("Output folder       :", OUTPUT_DIR)

# **4. Memuat Dataset**

Tahap ini memuat dataset dari folder `namadataset_raw`. Jika dataset masih berbentuk `.zip`, notebook akan mengekstraknya ke folder kerja `_working_dataset`. Jika dataset sudah berbentuk folder, notebook langsung mencari folder yang memuat subfolder kelas gambar.

In [ ]:
def list_raw_items(raw_dir: Path):
    """Menampilkan isi folder raw dataset."""
    items = sorted(raw_dir.iterdir()) if raw_dir.exists() else []
    print(f"Jumlah item di {raw_dir}: {len(items)}")
    for item in items:
        print("-", item.name)
    return items


def extract_zip_file(zip_path: Path, destination_dir: Path) -> Path:
    """Mengekstrak file zip ke folder tujuan."""
    extract_dir = destination_dir / zip_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)

    if not any(extract_dir.iterdir()):
        print(f"Mengekstrak {zip_path.name} ke {extract_dir} ...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(extract_dir)
    else:
        print(f"Folder ekstraksi sudah ada dan tidak kosong: {extract_dir}")

    return extract_dir


def prepare_dataset_source(raw_dir: Path, working_dir: Path) -> Path:
    """Menentukan sumber dataset, baik dari file zip maupun folder."""
    raw_items = list_raw_items(raw_dir)

    if not raw_items:
        raise FileNotFoundError(
            f"Folder {raw_dir} masih kosong. Letakkan dataset raw di folder ini terlebih dahulu."
        )

    archive_files = [item for item in raw_items if item.is_file() and item.suffix.lower() in SUPPORTED_ARCHIVE_EXTENSIONS]
    folder_items = [item for item in raw_items if item.is_dir()]

    if archive_files:
        return extract_zip_file(archive_files[0], working_dir)

    if folder_items:
        print(f"Menggunakan folder dataset raw: {folder_items[0]}")
        return folder_items[0]

    raise FileNotFoundError("Tidak ditemukan file .zip atau folder dataset yang valid di namadataset_raw.")


def count_images_inside(directory: Path) -> int:
    """Menghitung jumlah file gambar secara rekursif."""
    if not directory.exists():
        return 0
    return sum(1 for file in directory.rglob("*") if file.is_file() and file.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS)


def find_image_class_root(source_dir: Path) -> Path:
    """
    Mencari folder yang paling mungkin menjadi root kelas gambar.
    Root kelas adalah folder yang subfolder langsungnya berisi gambar.
    """
    candidates = []

    all_dirs = [source_dir] + [p for p in source_dir.rglob("*") if p.is_dir()]
    for directory in all_dirs:
        child_dirs = [child for child in directory.iterdir() if child.is_dir()]
        if len(child_dirs) < 2:
            continue

        class_like_dirs = []
        total_images = 0

        for child in child_dirs:
            image_count = sum(
                1 for file in child.iterdir()
                if file.is_file() and file.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS
            )
            if image_count > 0:
                class_like_dirs.append(child)
                total_images += image_count

        if len(class_like_dirs) >= 2:
            candidates.append((directory, len(class_like_dirs), total_images))

    if not candidates:
        raise FileNotFoundError(
            "Tidak ditemukan folder kelas gambar. Pastikan dataset memiliki subfolder kelas yang berisi gambar."
        )

    candidates = sorted(candidates, key=lambda item: (item[1], item[2]), reverse=True)
    return candidates[0][0]

In [ ]:
dataset_source = prepare_dataset_source(RAW_DIR, WORKING_DIR)
image_root = find_image_class_root(dataset_source)

class_names = sorted([
    path.name for path in image_root.iterdir()
    if path.is_dir() and count_images_inside(path) > 0
])

print("Dataset source      :", dataset_source)
print("Image class root    :", image_root)
print("Jumlah kelas        :", len(class_names))
print("Contoh nama kelas   :", class_names[:10])

# **5. Exploratory Data Analysis (EDA)**

EDA untuk dataset gambar dilakukan untuk memahami karakteristik dataset sebelum preprocessing. Analisis mencakup jumlah gambar per kelas, ketidakseimbangan kelas, contoh gambar, ukuran gambar, format file, dan validitas file gambar.

In [ ]:
def build_image_dataframe(image_root: Path) -> pd.DataFrame:
    """Membuat dataframe berisi path gambar dan label kelas."""
    records = []

    for class_dir in sorted([path for path in image_root.iterdir() if path.is_dir()]):
        label = class_dir.name
        for image_path in class_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS:
                records.append({
                    "image_path": str(image_path),
                    "label": label,
                    "extension": image_path.suffix.lower(),
                    "file_size_kb": image_path.stat().st_size / 1024,
                })

    return pd.DataFrame(records)

image_df = build_image_dataframe(image_root)

print("Jumlah total gambar:", len(image_df))
print("Jumlah kelas       :", image_df["label"].nunique())
image_df.head()

In [ ]:
class_distribution = (
    image_df["label"]
    .value_counts()
    .rename_axis("class_name")
    .reset_index(name="image_count")
)

class_distribution

In [ ]:
plt.figure(figsize=(12, max(6, len(class_distribution) * 0.35)))
plt.barh(class_distribution["class_name"], class_distribution["image_count"])
plt.title("Distribusi Jumlah Gambar per Kelas")
plt.xlabel("Jumlah Gambar")
plt.ylabel("Kelas")
plt.tight_layout()
plt.show()

In [ ]:
print("Ringkasan jumlah gambar per kelas")
print("Kelas terbanyak :", class_distribution.iloc[0]["class_name"], "=", class_distribution.iloc[0]["image_count"])
print("Kelas tersedikit:", class_distribution.iloc[-1]["class_name"], "=", class_distribution.iloc[-1]["image_count"])
print("Rata-rata gambar per kelas:", round(class_distribution["image_count"].mean(), 2))
print("Median gambar per kelas   :", round(class_distribution["image_count"].median(), 2))

In [ ]:
extension_distribution = (
    image_df["extension"]
    .value_counts()
    .rename_axis("extension")
    .reset_index(name="count")
)

extension_distribution

In [ ]:
def show_sample_images(image_df: pd.DataFrame, samples_per_class: int = 1, max_classes: int = 12):
    """Menampilkan contoh gambar dari beberapa kelas."""
    selected_classes = sorted(image_df["label"].unique())[:max_classes]
    sample_rows = []

    for label in selected_classes:
        label_df = image_df[image_df["label"] == label]
        rows = label_df.sample(n=min(samples_per_class, len(label_df)), random_state=SEED)
        sample_rows.extend(rows.to_dict("records"))

    total_images = len(sample_rows)
    cols = min(4, total_images)
    rows = int(np.ceil(total_images / cols))

    plt.figure(figsize=(cols * 4, rows * 4))

    for idx, row in enumerate(sample_rows, start=1):
        image = Image.open(row["image_path"]).convert("RGB")
        plt.subplot(rows, cols, idx)
        plt.imshow(image)
        plt.title(row["label"], fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_sample_images(image_df, samples_per_class=1, max_classes=12)

In [ ]:
def collect_image_size_info(image_df: pd.DataFrame, max_samples: int = 1000) -> pd.DataFrame:
    """Mengambil informasi ukuran gambar dari sampel dataset."""
    sample_df = image_df.sample(n=min(max_samples, len(image_df)), random_state=SEED)
    records = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Mengecek ukuran gambar"):
        image_path = Path(row["image_path"])
        try:
            with Image.open(image_path) as image:
                records.append({
                    "image_path": str(image_path),
                    "label": row["label"],
                    "width": image.width,
                    "height": image.height,
                    "mode": image.mode,
                })
        except Exception as error:
            records.append({
                "image_path": str(image_path),
                "label": row["label"],
                "width": None,
                "height": None,
                "mode": None,
                "error": str(error),
            })

    return pd.DataFrame(records)

size_df = collect_image_size_info(image_df, max_samples=1000)
size_df.describe(include="all")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(size_df["width"], size_df["height"], alpha=0.5)
plt.title("Sebaran Ukuran Gambar")
plt.xlabel("Width")
plt.ylabel("Height")
plt.tight_layout()
plt.show()

## Validasi Gambar Rusak

Validasi dilakukan untuk memastikan file gambar dapat dibuka oleh PIL. File kosong, file corrupt, atau file yang tidak bisa diverifikasi akan dipisahkan dari dataset yang siap diproses.

In [ ]:
def validate_image_file(image_path: Path):
    """Mengecek apakah satu file gambar valid atau rusak."""
    try:
        if image_path.stat().st_size == 0:
            return False, "empty_file"

        with Image.open(image_path) as image:
            image.verify()

        with Image.open(image_path) as image:
            image.convert("RGB")

        return True, None
    except Exception as error:
        return False, str(error)


def validate_all_images(image_df: pd.DataFrame) -> pd.DataFrame:
    """Mengecek semua gambar dan mengembalikan dataframe hasil validasi."""
    records = []

    for _, row in tqdm(image_df.iterrows(), total=len(image_df), desc="Validasi gambar"):
        image_path = Path(row["image_path"])
        is_valid, error_message = validate_image_file(image_path)
        records.append({
            "image_path": str(image_path),
            "label": row["label"],
            "is_valid": is_valid,
            "error_message": error_message,
        })

    return pd.DataFrame(records)

validation_df = validate_all_images(image_df)
validation_summary = validation_df["is_valid"].value_counts().rename_axis("is_valid").reset_index(name="count")
validation_summary

In [ ]:
valid_image_df = image_df.merge(
    validation_df[["image_path", "is_valid", "error_message"]],
    on="image_path",
    how="left"
)

valid_image_df = valid_image_df[valid_image_df["is_valid"] == True].copy()
corrupt_image_df = validation_df[validation_df["is_valid"] == False].copy()

print("Jumlah gambar valid :", len(valid_image_df))
print("Jumlah gambar rusak :", len(corrupt_image_df))

if len(corrupt_image_df) > 0:
    display(corrupt_image_df.head())

# **6. Data Preprocessing**

Preprocessing pada dataset gambar dilakukan dengan membagi file gambar valid ke dalam tiga subset: `train`, `val`, dan `test`. Pembagian dilakukan per kelas agar semua kelas tetap memiliki representasi pada setiap subset.

Rasio yang digunakan:

```text
Train      : 80%
Validation : 10%
Test       : 10%
```

Output preprocessing akan disimpan ke folder:

```text
preprocessing/namadataset_preprocessing
├── train
├── val
├── test
├── labels.txt
└── dataset_metadata.json
```

In [ ]:
def reset_output_directory(output_dir: Path):
    """Menghapus output lama dan membuat folder output baru."""
    if output_dir.exists():
        shutil.rmtree(output_dir)

    for split in ["train", "val", "test"]:
        (output_dir / split).mkdir(parents=True, exist_ok=True)


def split_class_images(class_df: pd.DataFrame):
    """Membagi data satu kelas menjadi train, val, test."""
    paths = class_df["image_path"].tolist()

    if len(paths) < 3:
        raise ValueError(
            f"Kelas {class_df['label'].iloc[0]} memiliki gambar kurang dari 3. "
            "Jumlah ini tidak cukup untuk train/val/test."
        )

    train_paths, temp_paths = train_test_split(
        paths,
        train_size=TRAIN_RATIO,
        random_state=SEED,
        shuffle=True,
    )

    val_relative_ratio = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_paths, test_paths = train_test_split(
        temp_paths,
        train_size=val_relative_ratio,
        random_state=SEED,
        shuffle=True,
    )

    return train_paths, val_paths, test_paths


def copy_images_to_split(paths, split_name: str, label: str, output_dir: Path):
    """Menyalin gambar ke folder split dan kelas yang sesuai."""
    target_class_dir = output_dir / split_name / label
    target_class_dir.mkdir(parents=True, exist_ok=True)

    copied_paths = []
    for source_path in paths:
        source_path = Path(source_path)
        target_path = target_class_dir / source_path.name

        counter = 1
        while target_path.exists():
            target_path = target_class_dir / f"{source_path.stem}_{counter}{source_path.suffix}"
            counter += 1

        shutil.copy2(source_path, target_path)
        copied_paths.append(str(target_path))

    return copied_paths


def run_image_preprocessing(valid_image_df: pd.DataFrame, output_dir: Path):
    """Menjalankan pipeline preprocessing gambar."""
    reset_output_directory(output_dir)

    split_records = []
    labels = sorted(valid_image_df["label"].unique())

    for label in tqdm(labels, desc="Membagi dataset per kelas"):
        class_df = valid_image_df[valid_image_df["label"] == label].copy()
        train_paths, val_paths, test_paths = split_class_images(class_df)

        split_map = {"train": train_paths, "val": val_paths, "test": test_paths}

        for split_name, paths in split_map.items():
            copied_paths = copy_images_to_split(paths, split_name, label, output_dir)
            for path in copied_paths:
                split_records.append({"split": split_name, "label": label, "image_path": path})

    return pd.DataFrame(split_records)

split_df = run_image_preprocessing(valid_image_df, OUTPUT_DIR)
print("Preprocessing selesai.")
split_df.head()

In [ ]:
split_summary = (
    split_df
    .groupby(["split", "label"])
    .size()
    .reset_index(name="image_count")
)

pivot_summary = split_summary.pivot_table(
    index="label",
    columns="split",
    values="image_count",
    fill_value=0,
).reset_index()

for column in ["train", "val", "test"]:
    if column not in pivot_summary.columns:
        pivot_summary[column] = 0

pivot_summary["total"] = pivot_summary[["train", "val", "test"]].sum(axis=1)
pivot_summary

In [ ]:
total_per_split = split_df["split"].value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)

plt.figure(figsize=(7, 5))
plt.bar(total_per_split.index, total_per_split.values)
plt.title("Jumlah Gambar Setelah Preprocessing")
plt.xlabel("Subset")
plt.ylabel("Jumlah Gambar")
plt.tight_layout()
plt.show()

total_per_split

## Menyimpan Label dan Metadata Dataset

File `labels.txt` digunakan agar proses modelling dapat membaca urutan kelas secara konsisten. File `dataset_metadata.json` digunakan sebagai bukti ringkas bahwa dataset hasil preprocessing sudah dibuat secara sistematis.

In [ ]:
def save_labels(labels, output_dir: Path):
    labels_path = output_dir / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as file:
        for label in labels:
            file.write(f"{label}\n")
    return labels_path


def save_metadata(output_dir: Path, image_root: Path, image_df: pd.DataFrame, validation_df: pd.DataFrame, split_df: pd.DataFrame):
    metadata = {
        "student_name": STUDENT_NAME,
        "dataset_name": DATASET_NAME,
        "image_root": str(image_root),
        "output_dir": str(output_dir),
        "seed": SEED,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "total_raw_images": int(len(image_df)),
        "total_valid_images": int(validation_df["is_valid"].sum()),
        "total_corrupt_images": int((validation_df["is_valid"] == False).sum()),
        "total_classes": int(split_df["label"].nunique()),
        "classes": sorted(split_df["label"].unique().tolist()),
        "split_counts": {split: int(count) for split, count in split_df["split"].value_counts().to_dict().items()},
        "image_size_for_modelling": IMG_SIZE,
        "batch_size_for_modelling": BATCH_SIZE,
    }

    metadata_path = output_dir / "dataset_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=4, ensure_ascii=False)

    return metadata_path, metadata

labels = sorted(split_df["label"].unique().tolist())
labels_path = save_labels(labels, OUTPUT_DIR)
metadata_path, metadata = save_metadata(OUTPUT_DIR, image_root, image_df, validation_df, split_df)

print("Labels disimpan ke  :", labels_path)
print("Metadata disimpan ke:", metadata_path)
metadata

# **7. Pemeriksaan Akhir Dataset Hasil Preprocessing**

Tahap ini memastikan struktur folder hasil preprocessing sudah benar dan siap digunakan pada proses training model.

In [ ]:
def inspect_preprocessed_dataset(output_dir: Path):
    """Menampilkan struktur ringkas dataset preprocessing."""
    for split in ["train", "val", "test"]:
        split_dir = output_dir / split
        class_dirs = sorted([path for path in split_dir.iterdir() if path.is_dir()]) if split_dir.exists() else []
        image_count = count_images_inside(split_dir) if split_dir.exists() else 0
        print(f"{split:<5} | kelas: {len(class_dirs):>3} | gambar: {image_count:>6}")

    print("labels.txt ada            :", (output_dir / "labels.txt").exists())
    print("dataset_metadata.json ada :", (output_dir / "dataset_metadata.json").exists())

inspect_preprocessed_dataset(OUTPUT_DIR)

In [ ]:
if TENSORFLOW_AVAILABLE:
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_dataset = tf.keras.utils.image_dataset_from_directory(
        TEST_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    print("Dataset berhasil dibaca oleh TensorFlow.")
    print("Class names:", train_dataset.class_names)
else:
    print("TensorFlow tidak tersedia. Pemeriksaan TensorFlow dilewati.")

In [ ]:
if TENSORFLOW_AVAILABLE:
    plt.figure(figsize=(10, 10))

    for images, labels_batch in train_dataset.take(1):
        for i in range(min(9, len(images))):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(train_dataset.class_names[int(labels_batch[i])])
            plt.axis("off")

    plt.tight_layout()
    plt.show()

# **8. Kesimpulan Eksperimen Preprocessing**

Notebook ini telah menjalankan tahap eksperimen dataset gambar secara manual. Tahapan yang sudah dilakukan meliputi data loading, EDA, validasi gambar, preprocessing pembagian dataset, serta penyimpanan label dan metadata dataset.

Dataset hasil preprocessing telah tersedia dalam struktur berikut:

```text
preprocessing/namadataset_preprocessing
├── train
├── val
├── test
├── labels.txt
└── dataset_metadata.json
```

Struktur tersebut sudah siap digunakan pada tahap berikutnya, yaitu pembangunan model machine learning atau deep learning menggunakan MLflow. Logika preprocessing pada notebook ini dapat dikonversi ke file `automate_Nama-siswa.py` agar proses preprocessing dapat dijalankan otomatis melalui GitHub Actions.